# Test behavior rather than printed output

QuickCart counts tickets by support category. If one ticket repeats a tag, that must not make it look like more tickets need attention.

> **Your goal:** Test what a data transformation means for the business, not just what it prints.<br>
> **You need:** Functions, collections and the validated `Ticket` idea from earlier notebooks.<br>
> **Effort:** 30–45 minutes (estimate).<br>
> **You will produce:** A test that catches the duplicate-tag bug, evidence of the fix, and tests for selecting priority tickets.

## Before executing

Work in your own copy, in a notebook editor running Python 3.11. Running it in Colab has not been tested, and hosted notebooks may use a different Python version. You need no extra packages, keys, paid model or real customer data. Keep your work and notes private. Nothing here starts a server or uses a paid API; stop your kernel when you finish.

Start from the cells as supplied, not from an old kernel. Predict, run the cells in order, add the fault described, restore the rule and rerun the same check. Restart before your final in-order run. The notebook starts with all outputs cleared. Note that an unfinished TODO function can sit there while "Run all" still passes; you must call your own function separately to test it.

[Previous: async and exceptions](03-async-and-exceptions.ipynb) · [Bridge route](README.md) · [Concept explanation](../../study-guides/python-engineering-bridge/README.md)

## Predict the business meaning

QuickCart wants to know how many **tickets** mention each category, not how many times a tag appears. Predict the summary for these made-up tickets:

| Ticket | Amount (teaching units) | Tags |
|---|---|---|
| A | 10 | shipping, shipping |
| B | 20 | shipping, returns |

What are the ticket count, the total amount, the shipping count and the returns count? Which test would catch code that looks reasonable but answers the wrong question?

<details>
<summary>After predicting: compare the calculation</summary>

There are **2 tickets**, totalling **10 + 20 = 30 units**. Shipping appears on **2 tickets** and returns on **1**. Counting three shipping mentions would exaggerate how many tickets need attention.

</details>

## Read the transformation in stages

The `Ticket` model below is supplied from the validation lesson. `summarize` expects tickets that are already validated; it does not check who is calling or reject duplicate ticket IDs across a batch.

1. The set comprehension collects each category label once, across all tickets.
2. `sorted` puts the labels in a predictable order.
3. `len` counts the tickets and `sum` adds up their amounts.
4. For each category, the code asks once per ticket "does this ticket have the tag?". `True` adds 1 and `False` adds 0. Using `True`/`False` for counting is fine here, even though they are not allowed as amounts.

Use comprehensions when they keep a transformation readable. Switch to an ordinary loop when validation, error handling or intermediate steps make it easier to explain. Shorter code is not automatically clearer.

## Run behavior checks

The test cell uses `unittest.TestCase.assertEqual` to compare what the code really returns with the value you declared. It checks the duplicate-tag case and an empty batch. The final `assert run.wasSuccessful()` makes the notebook cell fail if any test fails; just printing a summary would not prove anything.

After running the supplied tests, add a test in your copy for the two-ticket example above, checking all four values. After editing, rerun the class definition and reload the suite, because an old suite can still hold the old tests.

In [ ]:
from dataclasses import dataclass


class InputError(ValueError):
    """A declared synthetic ticket contract failed; messages never echo raw payloads."""


@dataclass(frozen=True)
class Ticket:
    ticket_id: str
    amount: int
    tags: tuple[str, ...] = ()

    def __post_init__(self) -> None:
        if not isinstance(self.ticket_id, str) or not self.ticket_id.strip() or len(self.ticket_id) > 64:
            raise InputError('Ticket ID must be nonempty text of at most 64 characters')
        if type(self.amount) is not int or not 0 <= self.amount <= 1_000_000:
            raise InputError('Amount must be an integer in synthetic units, 0–1000000')
        if not isinstance(self.tags, tuple) or len(self.tags) > 10:
            raise InputError('Tags must be a tuple with at most ten values')
        if any(not isinstance(tag, str) or not tag.strip() or len(tag) > 40 for tag in self.tags):
            raise InputError('Tags must be nonempty strings of at most 40 characters')




In [ ]:
def summarize(tickets: list[Ticket]) -> dict[str, object]:
    # One ticket contributes at most once per tag; annotations alone do not enforce this rule.
    tags = sorted({tag for ticket in tickets for tag in ticket.tags})
    return {'count': len(tickets), 'total_units': sum(ticket.amount for ticket in tickets),
            'by_tag': {tag: sum(tag in ticket.tags for ticket in tickets) for tag in tags}}


In [ ]:
import unittest
class SummaryTests(unittest.TestCase):
    def test_duplicate_tag_is_one_ticket(self):
        self.assertEqual(summarize([Ticket("A",10,("shipping","shipping"))])["by_tag"], {"shipping":1})
    def test_empty_batch(self):
        self.assertEqual(summarize([]), {"count":0,"total_units":0,"by_tag":{}})
suite = unittest.defaultTestLoader.loadTestsFromTestCase(SummaryTests)
run = unittest.TextTestRunner().run(suite)
assert run.wasSuccessful()


## Reproduce a meaningful regression

In your copy of `summarize`, replace:

```python
sum(tag in ticket.tags for ticket in tickets)
```

with:

```python
sum(ticket.tags.count(tag) for ticket in tickets)
```

Rerun the function cell, then the test cell. The duplicate-tag test should now fail: the broken version reports shipping as 2 for a single ticket instead of 1. The empty-batch test may still pass, which shows why testing only the easy or empty cases is not enough.

Restore the "does this ticket have the tag?" counting and rerun the same tests. Keep the evidence of the failure and the fix; removing the duplicate tags from the test data would just hide the bug.

## Independent task: select priority tickets

You can practise this in your own notebook function first if that helps, then build the real version in `exercises/priority.py` at the required [package checkpoint](python-bridge/README.md). Follow that project's setup before running its commands; do not create a second, unrelated application.

**The rules:** take validated tickets and a threshold that is a whole number of zero or more, never `True` or `False`. Select tickets whose amount is **at or above** the threshold, order them by amount from highest to lowest and then by ID for ties, return their IDs, and leave the input list unchanged. Reject an invalid threshold with `ValueError`.

Write your tests before you look at the reference. These examples describe what should happen, not how to code it:

```python
tickets = [Ticket("B", 30), Ticket("C", 20), Ticket("A", 30)]
before = list(tickets)
assert priority_ids(tickets, 20) == ["A", "B", "C"]
assert priority_ids(tickets, 31) == []
assert priority_ids([], 0) == []
assert tickets == before

for threshold in (True, -1, "20", 2.5, None):
    try:
        priority_ids(tickets, threshold)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid threshold accepted")
```

If `priority_ids` is not defined yet, write your function before running these examples. Add one case of your own and explain the risk it catches.

<details>
<summary>After your attempt: hints and reasoning</summary>

Check the threshold first, even when the list is empty. Build a new sorted list so the caller's list keeps its order. A sort key of (negative amount, ID) gives highest amount first with IDs in order for ties; reversing the whole sort would also reverse the ID order. Keep the "or equal" in the filter: a ticket exactly at the threshold counts.

</details>

In the package project's `python-bridge` folder, after its setup, run `uv run python exercises/check_priority.py` against your function. The reference answer sits next to the starter file; say so if you consult it. The package lesson also checks that the installed wheel works outside the source folder, which no notebook run can prove.

## Troubleshoot and reflect

| Symptom | Check first | Fix |
|---|---|---|
| Your edited test does not seem to run | The test class and suite in memory | Rerun the class definition and reload the suite. |
| Asserts pass, but you only looked at printed output | Whether the test checks the actual requirement | Assert on the real returned value, including failure cases. |
| Tied priorities come out in the wrong order | The sort direction of each part of the key | Test two equal amounts with IDs deliberately out of order. |
| The input list changes | Whether sorting changed the caller's list | Compare with a saved copy and return a separately sorted list. |
| `NameError` | Definitions and cell order | Run the earlier cells; restart if you are unsure what is in memory. |

**Reflect:** which test caught the wrong business meaning? Which requirement is still untested? Keep your first prediction, the failure, your fix, your own results and any help used, privately. Clear sensitive output before sharing and stop your kernel when you finish; nothing here starts a server or costs money.

## Check understanding and finish

**Pause and decide:** one ticket lists shipping twice; a second ticket lists shipping once. What should the shipping count be?

<details>
<summary>Compare your reasoning</summary>

Two tickets. Counting three mentions answers a different question; keep the "count tickets" rule and test the repeated-tag case.

</details>

Before you call this done, keep:

- Your prediction and the real reference result from a fresh, in-order run.
- The deliberately failing check, your fix and the check passing again.
- Asserts that really call your own function, including invalid input.
- A short explanation of the boundary, and any hints or AI help you used.

**You can now:** test what a data transformation means for the business, and prove it with checks rather than a saved green result.

**Next:** complete the [required package checkpoint](python-bridge/README.md), after any other preparation your [bridge map](README.md#unit-map-and-estimated-effort) selected. The package checkpoint is required before the API project. Save your evidence, clear sensitive output before sharing and stop your session.